This notebook prepares the raw corpus for downstream NLP tasks. It covers: data loading and light cleaning, Italian sentence segmentation, keyword-based filtering for two topics (migration and defence), and selection of the top speeches per group. The output files are pickled DataFrames consumed by the translation and summarization notebook.


# Setting up the Environment

In [ ]:
!pip install spacy sentence-transformers
!python -m spacy download it_core_news_sm #since the text is in italian
!pip install "optimum[onnxruntime]"   # speeds up cross-encoder (not used in the end)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 138.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('it_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


"Version conflicts between peft, transformers, and sentence-transformers required a downgrade. The cell below uninstalls and reinstalls compatible versions."

In [ ]:
!pip uninstall -y peft transformers sentence-transformers
!pip install -q "transformers<4.40" "sentence-transformers"

Found existing installation: peft 0.19.1
Uninstalling peft-0.19.1:
  Successfully uninstalled peft-0.19.1
Found existing installation: transformers 4.57.6
Uninstalling transformers-4.57.6:
  Successfully uninstalled transformers-4.57.6
Found existing installation: sentence-transformers 5.4.1
Uninstalling sentence-transformers-5.4.1:
  Successfully uninstalled sentence-transformers-5.4.1
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 89.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.3/245.3 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 103.0 MB/s eta 0:00:00


In [ ]:
import re
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer, CrossEncoder
from sklearn.metrics.pairwise import cosine_similarity

/usr/local/lib/python3.12/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


# Data Loading and Preprocessing

The UN dataset (ITA_speeches.xlsx) was assembled manually by stacking the Italian-language speeches delivered to the UN General Debate from 2006 to 2022. The source is the United Nations General Debate Corpus. Since there is one speech per year, the dataset is small (17 rows) and does not require the same grouping logic applied to the parliamentary corpus.

The speeches are taken from the United Nations General Debate Corpus (https://www.ungdc.bham.ac.uk/?utm_source)-


In [ ]:
UN = pd.read_excel('ITA_speeches.xlsx')

The parliamentary dataset is taken from the ItaParlCorpus database (https://dataverse.harvard.edu/dataset.xhtml?persistentId=doi:10.7910/DVN/KUARWD) and contains all the parliamentary speeches from 2006 to 2022 of the Camera dei Deputati.

In [ ]:
cols_to_keep = ["row_id","speaker", "text", "party_family", 'year']

df = pd.read_csv('camera_2006-2022.csv', usecols=cols_to_keep, engine='python')

# Drop rows with missing text
df = df.dropna(subset=["text"]).copy()

print(df.shape)
df.head(-20)

we check that there is data for each year. This is fundamental because we will groupby year at the end of this preparation. It is also important because the dataset is 600MB, and we want to verify that the full dataset was imported before proceeding

The text is cleaned lightly: lowercasing, collapsing whitespace, and normalising quotation marks. A deeper cleaning (e.g. stop-word removal, stemming) would add complexity that is not justified at this stage given the volume of rows and the nature of the downstream tasks



In [ ]:

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"\s+", " ", text)              # collapse spaces
    text = re.sub(r"[“”]", '"', text)
    text = re.sub(r"[‘’]", "'", text)
    return text.strip()

df["text_clean"] = df["text"].apply(clean_text)

# Sentencizer
We split the speech in sentences, which will become our unit of analysis for some tasks.

**Build_italian_sentencizer** returns a spaCy pipeline with the Italian sentencizer. It uses *'it_core_news_sm'* which does not need a parser, making it fast.

Why this matters: an English sentencizer would have missed Italian
constructs like long parliamentary periods, quotations introduced
by «...», and incidental clauses. The Italian model handles these
correctly, which reduces over-splitting and recovers sentences
that would have been discarded as fragments.

**resegment_corpus** Re-runs sentence splitting on the full corpus with the Italian model. It returns a sentence-level dataframe with one row per sentence.



In [ ]:

# ─────────────────────────────────────────────────────────────────────────────
# MODULE 1 — Italian sentence segmentation (replaces spacy.blank("en"))
# ─────────────────────────────────────────────────────────────────────────────

def build_italian_sentencizer():
    import spacy
    nlp = spacy.load("it_core_news_sm", exclude=["parser", "ner", "morphologizer"])
    # The small model includes a sentencizer via senter (dependency-free)
    # If senter is not available, fall back to rule-based sentencizer
    if "senter" not in nlp.pipe_names and "sentencizer" not in nlp.pipe_names:
        nlp.add_pipe("sentencizer")
    return nlp


def split_sentences_it(text, nlp):
    doc = nlp(text)
    return [sent.text.strip() for sent in doc.sents if len(sent.text.strip()) > 10]


def resegment_corpus(df, text_col="text_clean", nlp=None):

    if nlp is None:
        nlp = build_italian_sentencizer()

    df = df.copy()

    # Create safe unique speech id.
    # Row_id is not unique, it is a re-elaboration of the date, hence speech_id if more reliable
    if "speech_id" not in df.columns:
        df = df.reset_index(drop=True)
        df["speech_id"] = df.index

    # Safety checks
    if "year" not in df.columns:
        raise ValueError("Missing column: year")

    if text_col not in df.columns:
        raise ValueError(f"Missing column: {text_col}")

    df["sentences"] = df[text_col].apply(lambda t: split_sentences_it(t, nlp))

    df_sent = df.explode("sentences").rename(columns={"sentences": "sentence"})
    df_sent = df_sent.dropna(subset=["sentence"]).copy()

    # sentence number inside each original speech
    df_sent["sentence_id"] = df_sent.groupby("speech_id").cumcount()

    df_sent = df_sent.reset_index(drop=True)

    return df_sent

In [ ]:
print("=== Step 1: Italian sentence segmentation ===")
nlp = build_italian_sentencizer()
df = df.reset_index(drop=True)
df["speech_id"] = df.index
df_sent = resegment_corpus(df, text_col="text_clean", nlp=nlp)
print(f"  Total sentences: {len(df_sent):,}")
df_sent.to_pickle('/content/df_sent_full.pkl')

For UN we do the same. We only have one speech per year, hence speech_id is superfluous.

In [ ]:
def build_italian_sentencizer_un():

    import spacy
    nlp = spacy.load("it_core_news_sm", exclude=["parser", "ner", "morphologizer"])
    # The small model includes a sentencizer via senter (dependency-free)
    # If senter is not available, fall back to rule-based sentencizer
    if "senter" not in nlp.pipe_names and "sentencizer" not in nlp.pipe_names:
        nlp.add_pipe("sentencizer")
    return nlp


def split_sentences_it_un(text, nlp):
    """Drop-in replacement for the original split_sentences()."""
    doc = nlp(text)
    return [sent.text.strip() for sent in doc.sents if len(sent.text.strip()) > 10]


def resegment_corpus_un(df, text_col="text_clean", nlp=None):

    if nlp is None:
        nlp = build_italian_sentencizer_un()
    # Keep the original row index as speech_id
    df = df.reset_index().rename(columns={"index": "row_id"})
    df = df.copy()
    df["sentences"] = df[text_col].apply(lambda t: split_sentences_it_un(t, nlp))
    df_sent = df.explode("sentences").rename(columns={"sentences": "sentence"})
    df_sent = df_sent.dropna(subset=["sentence"]).copy()
    df_sent["sentence_id"] = df_sent.groupby("row_id").cumcount()
    df_sent = df_sent.reset_index(drop=True)
    return df_sent

In [ ]:
print("=== Step 1: Italian sentence segmentation ===")
nlp_un = build_italian_sentencizer_un()
df_sent_un = resegment_corpus_un(df_un, text_col="text_clean", nlp=nlp_un)
print(f"  Total sentences: {len(df_sent_un):,}")
df_sent_un.to_pickle('/content/df_sent_un.pkl')

=== Step 1: Italian sentence segmentation ===
  Total sentences: 1,417


# Filter
Each sentence is matched against a list of domain-specific keywords using a regex pattern with word boundaries. Sentences containing at least one keyword hit are retained. Two separate keyword lists are used: one for migration-related content and one for defence/war-related content. The same logic is applied to both the parliamentary and UN corpora, with language-adapted keyword lists (Italian for parliament, English for UN).

We initially attempted a two-stage semantic filter, a bi-encoder for candidate retrieval followed by a cross-encoder for reranking, to address lexical ambiguity (e.g. keywords that appear in unrelated political contexts)

However, as the process is costly and the results for a subset were not improving the filtering process we decided not to proceed.

In [ ]:

MIGRATION_KEYWORDS_EXTENDED = [
    # Core terms
    "migrante", "migranti", "immigrato", "immigrati", "emigrato", "emigrati",
    "immigrazione", "emigrazione", "migrazione", "flussi migratori", "flusso migratorio",
    # Legal status & procedures
    "richiedente asilo", "richiedenti asilo", "rifugiato", "rifugiati",
    "protezione internazionale", "protezione umanitaria", "protezione sussidiaria",
    "permesso di soggiorno", "permesso di residenza", "visto d'ingresso",
    "rimpatrio", "espulsione", "trattenimento",
    "centro di detenzione", "CPR", "CIE", "hotspot",
    "regolarizzazione", "sanatoria",
    "clandestino", "clandestini", "irregolare", "irregolari", "soggiornante irregolare",
    # Crossings & routes
    "sbarco", "sbarchi", "attraversamento", "frontiera", "confine",
    "rotta balcanica", "canale di sicilia", "mediterraneo centrale", "mediterraneo orientale",
    "barcone", "barconi", "gommone", "natante", "traversata",
    "naufragio", "naufragi", "disperso in mare",
    "operazione di salvataggio", "operazione di soccorso", "soccorso in mare",
    "zona SAR",
    # Actors & structures
    "unhcr", "oim", "guardia costiera", "guardia di frontiera",
    "frontex", "euaa", "commissione territoriale",
    "centro di accoglienza", "centro di prima accoglienza",
    "CAS", "SPRAR", "SAI", "struttura di accoglienza",
    "campo profughi", "campo di accoglienza",
    "nave ong", "nave umanitaria",
    "trafficante", "trafficanti di esseri umani", "tratta di esseri umani",
    # Policy & integration
    "politica migratoria", "politica dell'immigrazione",
    "quote migranti", "redistribuzione", "ricollocazione",
    "accordo di riammissione", "accordo di rimpatrio",
    "decreto flussi", "decreto sicurezza",
    "integrazione", "inclusione", "coesione sociale",
    "intercultura", "multiculturalismo", "accoglienza diffusa",
    # Demographics & labour
    "lavoratore straniero", "lavoratori stranieri", "manodopera straniera",
    "cittadino straniero", "cittadini stranieri", "comunità straniera",
    "minore straniero non accompagnato", "msna", "seconda generazione",
    "ricongiungimento familiare",
]

WAR_KEYWORDS_EXTENDED=[
    # Core
    "difesa", "difesa nazionale", "difesa europea", "difesa comune",
    "esercito", "forze armate", "militare", "militari",
    "forza militare", "capacità militare",
    # Branches & corps
    "esercito italiano", "marina militare", "aeronautica militare",
    "carabinieri", "guardia di finanza", "forze speciali",
    # Operations & missions
    "missione militare", "missione internazionale", "missione di pace",
    "operazione militare", "operazione nato", "teatro operativo",
    "teatro di guerra", "zona di conflitto", "area di crisi",
    "peacekeeping", "peacebuilding", "peace enforcement",
    "dispiegamento", "contingente", "contingente militare",
    "soldato", "soldati", "personale militare",
    "caduto in missione", "ferito in missione",
    # Alliances & organisations
    "nato", "alleanza atlantica", "patto atlantico", "articolo 5",
    "pesco", "cooperazione strutturata permanente",
    "agenzia europea per la difesa", "eda",
    "caschi blu", "missione onu", "unifil", "eufor", "eunavfor",
    "forza di reazione rapida",
    # Armaments & spending
    "spesa militare", "bilancio della difesa", "investimento nella difesa",
    "2% pil", "armamento", "armamenti", "sistema d'arma",
    "sistema missilistico", "drone militare", "veicolo blindato",
    "carro armato", "cacciabombardiere", "portaerei", "sottomarino",
    "fregate", "munizioni", "fornitura di armi", "esportazione di armi",
    "industria della difesa", "leonardo", "fincantieri",
    # Security & threats
    "sicurezza nazionale", "sicurezza collettiva", "deterrenza",
    "deterrenza nucleare", "minaccia alla sicurezza",
    "cyberattacco", "cybersicurezza", "guerra ibrida",
    "guerra asimmetrica", "guerra cibernetica",
    "intelligence", "servizi segreti", "AISE", "AISI", "DIS",
    "controspionaggio", "proliferazione nucleare", "arma nucleare", "disarmo",
    # Key conflicts in scope (2006-2022)
    "afghanistan", "iraq", "libia", "mali", "somalia",
    "libano", "kosovo", "bosnia", "ucraina", "donbass",
    "sahel", "siria", "crisi libica",
    "operazione sophia", "operazione atalanta", "missione eutm",
]


def compile_keyword_pattern(keywords):
    escaped = [re.escape(k.lower()) for k in keywords]
    pattern = r"\b(" + "|".join(escaped) + r")\b"
    return re.compile(pattern, flags=re.IGNORECASE)


def first_sentence_filter_extended(df_sent, keywords, sentence_col="sentence"):
    out = df_sent.copy()
    pattern = compile_keyword_pattern(keywords)
    out["keyword_hits"] = out[sentence_col].apply(
        lambda x: list({m.group(0).lower() for m in pattern.finditer(x.lower())})
    )
    out["n_keyword_hits"] = out["keyword_hits"].apply(len)
    out = out[out["n_keyword_hits"] > 0].copy()
    return out.reset_index(drop=True)

In [ ]:
print("\n===Extended keyword filter ===")
df_f1 = first_sentence_filter_extended(df_sent, MIGRATION_KEYWORDS_EXTENDED)
print(f"  After keyword filter: {len(df_f1):,}")
df_f1.to_pickle('/content/df_f1_m.pkl')

In [ ]:
print("\n===Extended keyword filter ===")
df_f1_w = first_sentence_filter_extended(df_sent, WAR_KEYWORDS_EXTENDED)
print(f"  After keyword filter: {len(df_f1_w):,}")
df_f1_w.to_pickle('/content/df_f1_w.pkl')

In [ ]:
# UN --> in english

MIGRATION_KEYWORDS_EXTENDED_UN = [
    # Core terms
    "migrant", "migrants", "immigrant", "immigrants", "emigrant", "emigrants",
    "immigration", "emigration", "migration", "migratory flows", "migratory flow",
    # Legal status & procedures
    "asylum seeker", "asylum seekers", "refugee", "refugees",
    "international protection", "humanitarian protection", "subsidiary protection",
    "residence permit", "entry visa",
    "repatriation", "deportation", "expulsion", "detention",
    "detention centre", "detention center", "hotspot",
    "regularization", "regularisation",
    "undocumented", "irregular migrant", "irregular migrants", "unauthorized migrant",
    # Crossings & routes
    "sea crossing", "border crossing", "crossing",
    "Balkan route", "Central Mediterranean", "Eastern Mediterranean",
    "boat", "rubber dinghy", "vessel",
    "shipwreck", "drowning at sea", "missing at sea",
    "search and rescue", "rescue operation", "rescue at sea",
    "SAR zone",
    # Actors & structures
    "UNHCR", "IOM", "coast guard", "border guard",
    "Frontex", "EUAA", "EASO",
    "reception centre", "reception center", "reception facility",
    "refugee camp", "humanitarian corridor",
    "NGO vessel", "humanitarian ship",
    "smuggler", "smugglers", "human trafficking", "traffickers",
    # Policy & integration
    "migration policy", "immigration policy",
    "refugee quota", "relocation", "redistribution",
    "readmission agreement", "return agreement",
    "integration", "inclusion", "social cohesion",
    "interculturalism", "multiculturalism",
    # Demographics & labour
    "foreign worker", "foreign workers", "migrant worker", "migrant workers",
    "foreign national", "foreign nationals",
    "unaccompanied minor", "unaccompanied minors", "unaccompanied children",
    "family reunification", "second generation",
    "remittances",
]

WAR_KEYWORDS_EXTENDED_UN = [
    # Core
    "defence", "defense", "national defence", "national defense",
    "European defence", "European defense", "common defence", "common defense",
    "armed forces", "military", "military forces", "military capacity",
    # Branches & corps
    "army", "navy", "air force", "special forces",
    # Operations & missions
    "military mission", "international mission", "peace mission",
    "military operation", "NATO operation", "theatre of operations",
    "theatre of war", "conflict zone", "crisis area",
    "peacekeeping", "peacebuilding", "peace enforcement",
    "deployment", "contingent", "military contingent",
    "soldier", "soldiers", "military personnel",
    "fallen in mission", "wounded in mission",
    # Alliances & organisations
    "NATO", "Atlantic Alliance", "Article 5",
    "PESCO", "permanent structured cooperation",
    "European Defence Agency", "EDA",
    "Blue Helmets", "UN mission", "UNIFIL", "EUFOR", "EUNAVFOR",
    "rapid reaction force",
    # Armaments & spending
    "military spending", "defence budget", "defense budget", "defence investment",
    "2% GDP", "armament", "armaments", "weapons system",
    "missile system", "military drone", "armoured vehicle",
    "tank", "fighter jet", "aircraft carrier", "submarine",
    "frigate", "ammunition", "arms supply", "arms export",
    "defence industry", "defense industry",
    # Security & threats
    "national security", "collective security", "deterrence",
    "nuclear deterrence", "security threat",
    "cyberattack", "cybersecurity", "hybrid warfare",
    "asymmetric warfare", "cyber warfare",
    "intelligence", "secret services", "counterintelligence",
    "nuclear proliferation", "nuclear weapon", "disarmament",
    # Key conflicts in scope (2006-2022)
    "Afghanistan", "Iraq", "Libya", "Mali", "Somalia",
    "Lebanon", "Kosovo", "Bosnia", "Ukraine", "Donbas",
    "Sahel", "Syria", "Libyan crisis",
    "Operation Sophia", "Operation Atalanta", "EUTM mission",
]

def compile_keyword_pattern_un(keywords):
    escaped = [re.escape(k.lower()) for k in keywords]
    pattern = r"\b(" + "|".join(escaped) + r")\b"
    return re.compile(pattern, flags=re.IGNORECASE)


def first_sentence_filter_extended_un(df_sent, keywords, sentence_col="sentence"):

    out = df_sent.copy()
    pattern = compile_keyword_pattern_un(keywords)
    out["keyword_hits"] = out[sentence_col].apply(
        lambda x: list({m.group(0).lower() for m in pattern.finditer(x.lower())})
    )
    out["n_keyword_hits"] = out["keyword_hits"].apply(len)
    out = out[out["n_keyword_hits"] > 0].copy()
    return out.reset_index(drop=True)

In [ ]:
print("\n=== Extended keyword filter ===")
df_f1_un = first_sentence_filter_extended_un(df_sent_un, MIGRATION_KEYWORDS_EXTENDED_UN)
print(f"  After keyword filter: {len(df_f1_un):,}")
df_f1_un.to_pickle('/content/df_f1_m_un.pkl')


=== Step 2: Extended keyword filter ===
  After keyword filter: 64

=== Step 3: True-context filter (unchanged) ===
  After true-context filter: 28


In [ ]:
print("\n=== Extended keyword filter ===")
df_f1_un_w = first_sentence_filter_extended_un(df_sent_un, WAR_KEYWORDS_EXTENDED_UN)
print(f"  After keyword filter: {len(df_f1_un_w):,}")
df_f1_un_w.to_pickle('/content/df_f1_w_un.pkl')

# Dataframe structure

We re-organized the dataframe, limiting the amount of rows to the top 100 speeches for every group (year x party_family). Speeches are ranked by the total number of keyword hits across all their constituent sentences (n_keyword_hits). This is fundamental in order to make sure subsequent operation stay feasible.

note: we selected top speeches instead of top sentences as we found it a better format for the tasks of topic selection and summarization due to the presence of a richer context.

In [ ]:
def get_top_speeches(df: pd.DataFrame, n: int = 100) -> pd.DataFrame:

    df = df.copy()

    # Exclude 'presidente' party group
    df = df[df["party_family"].str.lower() != "presidente"]

    results = []

    for (year, party), group in df.groupby(["year", "party_family"]):

        # Sort sentences by keyword score, best first
        group_sorted = group.sort_values("n_keyword_hits", ascending=False)

        seen_speeches = []
        seen_ids = set()

        for _, row in group_sorted.iterrows():
            sid = row["speech_id"]
            if sid not in seen_ids:
                seen_ids.add(sid)
                seen_speeches.append(row)
            if len(seen_speeches) == n:
                break

        available = len(seen_speeches)
        if available < n:
            print(
                f"[NOTE] year={year}, party_family={party}: "
                f"only {available} unique speech{'es' if available != 1 else ''} available "
                f"(requested {n})."
            )

        results.append(pd.DataFrame(seen_speeches))

    final = pd.concat(results, ignore_index=True)
    return final


# ───────────────────────────────────────────────────────────────────────
top_speeches = get_top_speeches(df_f1)

print(f"\nTotal speeches selected: {len(top_speeches)}")
print(top_speeches.groupby(["year", "party_family"]).size().to_string())
top_speeches.head()

In [ ]:
top_speeches.to_csv('top_speeches_m.csv', index=False)
top_speeches.to_pickle('top_speeches_m.pkl')

In [ ]:
top_speeches_w = get_top_speeches(df_f1_w)
top_speeches_w.to_csv('top_speeches_w.csv', index=False)
top_speeches_w.to_pickle('top_speeches_w.pkl')
top_speeches_w.shape

# Additional things we did not keep in the final DF

As we noticed that the filtered sentences contained noise, due to the ambiguous nature of language, we attempted to refine the process. First by doing a second filtering, in order to create a clean seed for the encoding process. Then by using this seed as a basis for a biencoder, faster to find semantically similiar sentences in the complete dataframe. Lastly, the top 5000 sentences selected by the biencoder were put in a cross-encoder to properly assess their semantic score.

As this is a computationally expensive process, it was executed on a subset to assess whether there was a noticeable improvment in data quality that would justify th expense. As it was not the case, the idea was then discarded.

In [ ]:

print("\n=== True-context filter ===")
MIGRATION_TRUE_CONTEXT = [
"migrante", "migranti", "immigrazione", "immigrato", "immigrati",
"rifugiato", "rifugiati", "richiedente asilo", "richiedenti asilo",
"protezione internazionale", "protezione umanitaria",
"sbarco", "sbarchi", "frontex", "rimpatrio", "espulsione",
"clandestino", "clandestini", "irregolare",
"centro di accoglienza", "hotspot", "CPR", "SPRAR", "SAI",
"tratta di esseri umani", "naufragio",
"decreto flussi", "decreto sicurezza",
"quote migranti", "ricollocazione", "redistribuzione"]

def has_true_context(text):
    t = str(text).lower()
    return any(c in t for c in MIGRATION_TRUE_CONTEXT)

df_f2 = df_f1[df_f1["sentence"].apply(has_true_context)].copy().reset_index(drop=True)
print(f"  After true-context filter: {len(df_f2):,}")
df_f2.to_pickle('/content/df_f2_m.pkl')


=== Step 2: Extended keyword filter ===
  After keyword filter: 1

=== Step 3: True-context filter (unchanged) ===
  After true-context filter: 0


In [ ]:
CROSS_ENCODER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
# Italian-specific alternative (slower but better):
# CROSS_ENCODER_MODEL = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"


def rerank_with_cross_encoder(
    candidates_df,
    seed_df,
    text_col_candidates="sentence",
    text_col_seed="sentence",
    model_name=CROSS_ENCODER_MODEL,
    top_n_pairs=5000,       # how many bi-encoder candidates to rerank
    final_threshold=0.0,    # cross-encoder score threshold (0 = keep all, raise to filter)
    batch_size=64,
):

    print(f"Loading cross-encoder: {model_name}")
    ce_model = CrossEncoder(model_name, max_length=512)

    # Work on top-N only to keep runtime manageable
    candidates_df = candidates_df.copy().reset_index(drop=True)
    if len(candidates_df) > top_n_pairs:
        print(f"  Truncating to top {top_n_pairs} bi-encoder candidates for reranking")
        candidates_df = candidates_df.head(top_n_pairs)

    seed_texts = seed_df[text_col_seed].astype(str).tolist()
    candidate_texts = candidates_df[text_col_candidates].astype(str).tolist()

    print(f"  Scoring {len(candidate_texts)} candidates × {len(seed_texts)} seeds in batches...")

    best_scores = np.full(len(candidate_texts), -np.inf)
    best_seed_idx = np.zeros(len(candidate_texts), dtype=int)

    # To avoid O(candidates × seeds) pairs, we use a smarter loop:
    # for each seed sentence, score all candidates at once, then take running max.
    for s_idx, seed_text in enumerate(seed_texts):
        pairs = [(seed_text, cand) for cand in candidate_texts]
        scores = ce_model.predict(pairs, batch_size=batch_size, show_progress_bar=False)
        improved = scores > best_scores
        best_scores[improved] = scores[improved]
        best_seed_idx[improved] = s_idx
        if (s_idx + 1) % 10 == 0:
            print(f"    Processed {s_idx + 1}/{len(seed_texts)} seed sentences")

    candidates_df["cross_encoder_score"] = best_scores
    candidates_df["matched_seed_cross"] = [seed_texts[i] for i in best_seed_idx]

    if final_threshold > 0:
        candidates_df = candidates_df[
            candidates_df["cross_encoder_score"] >= final_threshold
        ].copy()

    candidates_df = candidates_df.sort_values(
        "cross_encoder_score", ascending=False
    ).reset_index(drop=True)

    print(f"  Reranking complete. {len(candidates_df)} candidates retained.")
    return candidates_df

In [ ]:
def run_improved_pipeline(
    df_sent,
    df_f2, #second filter to create a seed
    bi_encoder_model="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    bi_encoder_threshold=0.65,       # slightly lower than before — cross-encoder will clean up
    rerank_top_n=5000,               # how many bi-encoder candidates pass to cross-encoder
    cross_encoder_threshold=0.0,     # set > 0 to hard-filter after reranking
    window=1,
    # context window for text_chunk (kept as metadata)
):

    print("\n=== Step 4: Bi-encoder semantic expansion ===")
    bi_model = SentenceTransformer(bi_encoder_model)
    all_texts = df_sent["sentence"].astype(str).tolist()
    seed_texts = df_f2["sentence"].astype(str).tolist()

    print("  Encoding all sentences...")
    all_emb = bi_model.encode(all_texts, convert_to_numpy=True, show_progress_bar=True, batch_size=256)
    print("  Encoding seed sentences...")
    seed_emb = bi_model.encode(seed_texts, convert_to_numpy=True, show_progress_bar=True)

    sim_matrix = cosine_similarity(all_emb, seed_emb)
    max_sim = sim_matrix.max(axis=1)
    best_seed_idx = sim_matrix.argmax(axis=1)

    seed_set = set(seed_texts)
    is_not_seed = df_sent["sentence"].astype(str).apply(lambda x: x not in seed_set)
    above_threshold = max_sim >= bi_encoder_threshold
    mask = is_not_seed & above_threshold

    df_semantic = df_sent[mask].copy()
    df_semantic["semantic_score"] = max_sim[mask]
    df_semantic["matched_seed_text"] = [seed_texts[idx] for idx in best_seed_idx[mask]]
    df_semantic["retrieval_type"] = "semantic_added"
    df_semantic = df_semantic.drop_duplicates(subset=["sentence"])

    df_semantic = df_semantic.sort_values("semantic_score", ascending=False).reset_index(drop=True)
    print(f"  After bi-encoder expansion: {len(df_semantic):,}")

    print("\n=== Step 5: Cross-encoder reranking ===")
    df_final = rerank_with_cross_encoder(
        candidates_df=df_semantic,
        seed_df=df_f2,
        top_n_pairs=rerank_top_n,
        final_threshold=cross_encoder_threshold,
    )
    print(f"  After reranking: {len(df_final):,}")
    # Add seed sentences back (they are already validated)
    df_seed_tagged = df_f2.copy()
    df_seed_tagged["retrieval_type"] = "seed"
    df_seed_tagged["semantic_score"] = 1.0
    df_seed_tagged["cross_encoder_score"] = 1.0
    df_seed_tagged["matched_seed_cross"] = df_seed_tagged["sentence"]

    df_final = pd.concat([df_seed_tagged, df_final], ignore_index=True)
    df_final = df_final.drop_duplicates(subset=["sentence"]).reset_index(drop=True)


    print(f"  df_semantic:  {len(df_semantic):,} rows")
    print(f"  df_final:     {len(df_final):,} rows")
    return {
        "df_semantic": df_semantic,
        "df_final": df_final,

    }

In [ ]:
results_migration = run_improved_pipeline(df_sent, df_f2)


=== Step 4: Bi-encoder semantic expansion ===


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

  Truncating to top 5000 bi-encoder candidates for reranking
  Scoring 5000 candidates × 819 seeds in batches...
    Processed 10/819 seed sentences
    Processed 20/819 seed sentences
    Processed 30/819 seed sentences
    Processed 40/819 seed sentences
    Processed 50/819 seed sentences
    Processed 60/819 seed sentences
    Processed 70/819 seed sentences
    Processed 80/819 seed sentences
    Processed 90/819 seed sentences
    Processed 100/819 seed sentences
    Processed 110/819 seed sentences
    Processed 120/819 seed sentences
    Processed 130/819 seed sentences
    Processed 140/819 seed sentences
    Processed 150/819 seed sentences
    Processed 160/819 seed sentences
    Processed 170/819 seed sentences
    Processed 180/819 seed sentences
    Processed 190/819 seed sentences
    Processed 200/819 seed sentences
    Processed 210/819 seed sentences
    Processed 220/819 seed sentences
    Processed 230/819 seed sentences
    Processed 240/819 seed sentences
    Proc

In [ ]:
final=results_migration['df_final']

In [ ]:
def run_improved_pipeline_2(
    raw_df,#in this one we do df_f2 within
    keyword,# original corpus dataframe
    text_col="text_clean",
    bi_encoder_model="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    bi_encoder_threshold=0.65,       # slightly lower than before — cross-encoder will clean up
    rerank_top_n=5000,               # how many bi-encoder candidates pass to cross-encoder
    cross_encoder_threshold=0.0,     # set > 0 to hard-filter after reranking
    window=1,
    # context window for text_chunk (kept as metadata)
):

    print("=== Step 1: Italian sentence segmentation ===")
    df_sent = raw_df


    print("\n=== Step 2: Extended keyword filter ===")
    df_f1 = first_sentence_filter_extended(df_sent, keyword )
    print(f"  After keyword filter: {len(df_f1):,}")

    print("\n=== Step 3: True-context filter (unchanged) ===")
    WAR_TRUE_CONTEXT =[
    "forze armate", "esercito", "militare", "militari",
    "nato", "alleanza atlantica", "articolo 5",
    "missione militare", "missione internazionale", "contingente",
    "spesa militare", "bilancio della difesa", "armamenti",
    "difesa europea", "difesa comune", "pesco",
    "peacekeeping", "caschi blu", "unifil",
    "deterrenza", "sicurezza nazionale",
    "cyberattacco", "cybersicurezza", "guerra ibrida",
    "operazione sophia", "operazione atalanta"]

    def has_true_context(text):
        t = str(text).lower()
        return any(c in t for c in WAR_TRUE_CONTEXT)

    df_f2 = df_f1[df_f1["sentence"].apply(has_true_context)].copy().reset_index(drop=True)
    print(f"  After true-context filter: {len(df_f2):,}")

    print("\n=== Step 4: Bi-encoder semantic expansion ===")
    bi_model = SentenceTransformer(bi_encoder_model)
    all_texts = df_sent["sentence"].astype(str).tolist()
    seed_texts = df_f2["sentence"].astype(str).tolist()

    print("  Encoding all sentences...")
    all_emb = bi_model.encode(all_texts, convert_to_numpy=True, show_progress_bar=True, batch_size=256)
    print("  Encoding seed sentences...")
    seed_emb = bi_model.encode(seed_texts, convert_to_numpy=True, show_progress_bar=True)

    sim_matrix = cosine_similarity(all_emb, seed_emb)
    max_sim = sim_matrix.max(axis=1)
    best_seed_idx = sim_matrix.argmax(axis=1)

    seed_set = set(seed_texts)
    is_not_seed = df_sent["sentence"].astype(str).apply(lambda x: x not in seed_set)
    above_threshold = max_sim >= bi_encoder_threshold
    mask = is_not_seed & above_threshold

    df_semantic = df_sent[mask].copy()
    df_semantic["semantic_score"] = max_sim[mask]
    df_semantic["matched_seed_text"] = [seed_texts[idx] for idx in best_seed_idx[mask]]
    df_semantic["retrieval_type"] = "semantic_added"
    df_semantic = df_semantic.drop_duplicates(subset=["sentence"])

    df_semantic = df_semantic.sort_values("semantic_score", ascending=False).reset_index(drop=True)
    print(f"  After bi-encoder expansion: {len(df_semantic):,}")

    print("\n=== Step 5: Cross-encoder reranking ===")
    df_final = rerank_with_cross_encoder(
        candidates_df=df_semantic,
        seed_df=df_f2,
        top_n_pairs=rerank_top_n,
        final_threshold=cross_encoder_threshold,
    )
    print(f"  After reranking: {len(df_final):,}")

    # Add seed sentences back (they are already validated)
    df_seed_tagged = df_f2.copy()
    df_seed_tagged["retrieval_type"] = "seed"
    df_seed_tagged["semantic_score"] = 1.0
    df_seed_tagged["cross_encoder_score"] = 1.0
    df_seed_tagged["matched_seed_cross"] = df_seed_tagged["sentence"]

    df_final = pd.concat([df_seed_tagged, df_final], ignore_index=True)
    df_final = df_final.drop_duplicates(subset=["sentence"]).reset_index(drop=True)



    print("\n=== Pipeline complete ===")
    print(f"  df_filtered1: {len(df_f1):,} rows")
    print(f"  df_filtered2: {len(df_f2):,} rows")
    print(f"  df_semantic:  {len(df_semantic):,} rows")
    print(f"  df_final:     {len(df_final):,} rows")
    return {
        "df_filtered_1": df_f1,
        "df_filtered_2": df_f2,
        "df_semantic": df_semantic,
        "df_final": df_final,
    }


In [ ]:
results_war = run_improved_pipeline_2(df_sent, WAR_KEYWORDS_EXTENDED)

=== Step 1: Italian sentence segmentation ===

=== Step 2: Extended keyword filter ===
  After keyword filter: 3,916

=== Step 3: True-context filter (unchanged) ===
  After true-context filter: 1,913

=== Step 4: Bi-encoder semantic expansion ===


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Encoding all sentences...


Batches:   0%|          | 0/569 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/60 [00:00<?, ?it/s]

  After bi-encoder expansion: 44,516

=== Step 5: Cross-encoder reranking ===
Loading cross-encoder: cross-encoder/ms-marco-MiniLM-L-6-v2


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Truncating to top 5000 bi-encoder candidates for reranking
  Scoring 5000 candidates × 1913 seeds in batches...
    Processed 10/1913 seed sentences
    Processed 20/1913 seed sentences
    Processed 30/1913 seed sentences
    Processed 40/1913 seed sentences
    Processed 50/1913 seed sentences
    Processed 60/1913 seed sentences
    Processed 70/1913 seed sentences
    Processed 80/1913 seed sentences
    Processed 90/1913 seed sentences
    Processed 100/1913 seed sentences
    Processed 110/1913 seed sentences
    Processed 120/1913 seed sentences
    Processed 130/1913 seed sentences
    Processed 140/1913 seed sentences
    Processed 150/1913 seed sentences
    Processed 160/1913 seed sentences
    Processed 170/1913 seed sentences
    Processed 180/1913 seed sentences
    Processed 190/1913 seed sentences
    Processed 200/1913 seed sentences
    Processed 210/1913 seed sentences
    Processed 220/1913 seed sentences
    Processed 230/1913 seed sentences
    Processed 240/191

In [ ]:
results_w=results_war['df_final']
semantic_w=results_war['df_semantic']

In [ ]:
df_f1_w=results_war['df_filtered_1']
df_f1_w.to_pickle('/content/df_f1_w.pkl')

In [ ]:
df_f2_w=results_war['df_filtered_2']
df_f2_w.to_pickle('/content/df_f2_w.pkl')